# Fase 4 · Ferramentas externas e RAG existente: ampliando o alcance do agente

**Projeto:** Assistente de Viagens com LangChain Agents
**Autora:** Shaiane Alves Tonin

| Requisito do enunciado | Seção |
|---|---|
| Ferramentas com nome, descrição e lógica central; hardcoded vs. semânticas | 1 |
| Integração por tipo de uso: web API, API interna (microsserviço), função serverless | 2 |
| Padrão text-to-query (SQL) | 3 |
| Agente de recuperação sobre o índice FAISS **existente**, sem reindexar | 4 |
| Casos difíceis e diagnóstico de falhas pelos documentos-fonte retornados | 5 |
| Transformação de consulta para corrigir a recuperação, com justificativa | 6, 7 |

In [1]:
import os, sys, json, time
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    os.chdir("/content/projeto-langchain-agentes")
    %pip install -q -r requirements.txt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
pd.set_option("display.max_colwidth", 160)

from viagens.config import USE_DUMMY, get_llm
from viagens.callbacks import LogExecucaoHandler
from viagens.banco import criar_banco

llm = get_llm(temperature=0)
log_handler = LogExecucaoHandler(sessao="fase4")
CFG = {"callbacks": [log_handler]}
criar_banco()   # recria o banco fictício (o microsserviço da seção 2 grava reservas nele)
print("Modo dummy:", USE_DUMMY)

/Users/Shaiane/projeto-langchain-agentes/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Modo dummy: False


## 1. As ferramentas do agente, documentadas por tipo

Dois eixos classificam cada ferramenta:

- **Lógica central: hardcoded ou semântica.** Uma função *hardcoded* tem lógica fixa: a mesma entrada sempre dá a mesma saída, e ela pode ser testada com `assert`. Uma função *semântica* depende da interpretação de um modelo (LLM ou embeddings): a saída varia com a formulação da entrada e precisa ser **avaliada** com casos de teste, não só testada.
- **Tipo de integração:** web API externa, API interna (microsserviço), função serverless, text-to-query (banco de dados) ou busca semântica (vector store).

Isso tem consequência prática: as ferramentas semânticas são as que precisam de salvaguardas, como a validação do SQL gerado (seção 3) e a transformação de consulta (seção 6).

In [2]:
from viagens.tools import ferramentas_estruturadas, ferramentas_fase4
from viagens.servicos import reservas_api

URL_RESERVAS = reservas_api.iniciar_servico()
tools = ferramentas_estruturadas() + ferramentas_fase4(llm, url_reservas=URL_RESERVAS)
por_nome = {t.name: t for t in tools}

CATALOGO = {
    "previsao_tempo":       ("hardcoded", "web API externa (Open-Meteo)", "geocodifica a cidade; previsão até 16 dias ou referência histórica do ano anterior"),
    "converter_moeda":      ("hardcoded", "web API externa (Frankfurter/BCE)", "converte com a cotação do dia; erro explícito para moeda não cotada"),
    "calculadora":          ("hardcoded", "função local", "aritmética segura via AST (sem eval)"),
    "buscar_regras_viagem": ("semântica", "busca semântica (FAISS existente)", "embedding da consulta e top-k trechos da base de regras"),
    "consultar_banco":      ("semântica", "text-to-query (SQLite)", "LLM gera SQL → validação → execução read-only → resposta"),
    "listar_reservas":      ("hardcoded", "API interna (microsserviço HTTP)", "GET /reservas: reservas do usuário com status"),
    "criar_reserva":        ("hardcoded", "API interna (microsserviço HTTP)", "POST /reservas: valida voo e assentos, cria reserva 'pendente'"),
    "estimar_custo":        ("hardcoded", "função serverless (contrato FaaS)", "passagens + hospedagem + gastos + IOF 3,5% + margem 10%"),
}
pd.DataFrame([{"ferramenta": n, "lógica": CATALOGO[n][0], "integração": CATALOGO[n][1],
               "lógica central": CATALOGO[n][2], "descrição lida pelo LLM": por_nome[n].description[:110] + "..."}
              for n in CATALOGO])

,ferramenta,lógica,integração,lógica central,descrição lida pelo LLM
0,previsao_tempo,hardcoded,web API externa (Open-Meteo),geocodifica a cidade; previsão até 16 dias ou referência histórica do ano anterior,"Consulta a previsão do tempo REAL (API Open-Meteo) de uma cidade: temperaturas máx./mín., chuva e condição por..."
1,converter_moeda,hardcoded,web API externa (Frankfurter/BCE),converte com a cotação do dia; erro explícito para moeda não cotada,"Converte um valor entre moedas com a cotação REAL do dia (API Frankfurter, Banco Central Europeu). Use códigos..."
2,calculadora,hardcoded,função local,aritmética segura via AST (sem eval),"Calculadora aritmética exata. Use para somar custos, dividir orçamentos, porcentagens...."
3,buscar_regras_viagem,semântica,busca semântica (FAISS existente),embedding da consulta e top-k trechos da base de regras,"Busca semântica na base de conhecimento de viagens (índice FAISS): regras de entrada e visto, seguro-viagem, b..."
4,consultar_banco,semântica,text-to-query (SQLite),LLM gera SQL → validação → execução read-only → resposta,"Consulta o banco de viagens (SQLite) em linguagem natural: voos da AeroSul (datas, preços, escalas), hotéis (e..."
5,listar_reservas,hardcoded,API interna (microsserviço HTTP),GET /reservas: reservas do usuário com status,API interna de reservas: lista as reservas do usuário com status atualizado....
6,criar_reserva,hardcoded,API interna (microsserviço HTTP),"POST /reservas: valida voo e assentos, cria reserva 'pendente'",API interna de reservas: cria uma reserva de voo PENDENTE. Use só quando o usuário pedir explicitamente para r...
7,estimar_custo,hardcoded,função serverless (contrato FaaS),"passagens + hospedagem + gastos + IOF 3,5% + margem 10%","Função serverless que estima o custo total da viagem em BRL (passagens, hospedagem, gastos locais, IOF de 3,5%..."


**Observação.** Mesmo as ferramentas hardcoded são **acionadas** semanticamente: é o LLM que decide chamá-las e preenche os argumentos a partir da descrição. A classificação acima é sobre a **lógica interna** da ferramenta. Por isso as descrições foram escritas como prompts: o que a ferramenta faz, quando usar, o formato dos argumentos e o que ela **não** cobre (por exemplo, "Não cobre ARS nem CLP").

## 2. Integração por tipo de uso

| Tipo | Ferramenta | Por que este tipo | Cuidados implementados |
|---|---|---|---|
| **Web API externa** | Open-Meteo, Frankfurter | Dados públicos que mudam com o tempo, mantidos por terceiros | Timeout, `APIError` com mensagem legível, sem chave no código |
| **API interna / microsserviço** | `reservas_api` (HTTP/JSON) | **Escrever** num sistema da empresa exige regras de negócio (assentos, limites, status inicial). O agente não escreve direto no banco: chama o serviço dono dos dados | Validação de corpo, códigos HTTP (400/404/409/422), reserva sempre `pendente` (um humano confirma o pagamento) |
| **Função serverless** | `estimar_custo` | Cálculo sem estado, chamado sob demanda e versionado à parte: o contrato típico de FaaS (Lambda / Cloud Functions) | Contrato `handler(event, context)`, validação do evento, sem estado entre chamadas |

> O microsserviço roda numa thread local (`http.server`) e a função serverless é chamada por um invocador local que respeita o contrato FaaS. É uma simulação fiel do **contrato** de cada integração, sem exigir infraestrutura de nuvem para a avaliação.

In [3]:
import requests
print("Web API externa →", por_nome["converter_moeda"].invoke({"valor": 100, "de": "EUR", "para": "BRL"}, config=CFG), "\n")

print("API interna →", URL_RESERVAS, requests.get(f"{URL_RESERVAS}/saude").json())
print("GET /reservas:", json.dumps(json.loads(por_nome["listar_reservas"].invoke({}, config=CFG))["reservas"][0], ensure_ascii=False))
print("POST /reservas (voo inexistente):", por_nome["criar_reserva"].invoke({"voo_id": 999999, "pessoas": 2}, config=CFG))
print("POST /reservas (válido):", por_nome["criar_reserva"].invoke({"voo_id": 101, "pessoas": 2}, config=CFG), "\n")

from viagens.servicos import funcao_custo
print("Serverless (1ª chamada):", por_nome["estimar_custo"].invoke(
    {"dias": 10, "pessoas": 2, "passagem_por_pessoa_brl": 4500, "diaria_hotel_brl": 700}, config=CFG))
print("Serverless (evento inválido):", funcao_custo.invocar({"pessoas": 2}))

Web API externa → {"valor": 100.0, "de": "EUR", "para": "BRL", "convertido": 625.95, "taxa": 6.2595, "data_cotacao": "2026-11-13", "fonte": "Frankfurter (BCE)"} 

API interna → http://127.0.0.1:52230 {'status': 'ok'}
GET /reservas: {"id": 1, "tipo": "voo", "status": "confirmada", "data_inicio": "2026-12-16", "data_fim": null, "pessoas": 2, "valor_total_brl": 9172.14, "voo": "AS1015", "origem": "GRU", "destino": "LIS", "hotel": null, "cidade_hotel": null}
POST /reservas (voo inexistente): Erro 404 da API de reservas: {"erro": "voo 999999 não existe"}
POST /reservas (válido): {"reserva_id": 5, "status": "pendente", "voo": "AS1101", "data": "2027-03-19", "pessoas": 2, "valor_total_brl": 5372.02, "observacao": "Reserva pendente: o pagamento é confirmado por um atendente."} 

Serverless (1ª chamada): {"statusCode": 200, "body": {"itens_brl": {"passagens": 9000.0, "hospedagem": 7000.0, "gastos_locais": 11000.0, "iof_3_5%": 630.0, "margem_cambial_10%": 1800.0}, "total_brl": 29430.0, "por_pess

## 3. Text-to-query: perguntas em linguagem natural → SQL

O banco `data/viagens.db` (fictício e determinístico) tem 4 tabelas: `aeroportos`, `voos` (cerca de 3 mil voos da AeroSul, de dez/2026 a jun/2027), `hoteis` e `reservas`.

```
pergunta ──LLM (schema + tabela IATA + few-shot)──▶ SQL
        ──validar_sql: 1 instrução, só SELECT, sem escrita/sistema, LIMIT──▶
        ──validar_coerencia: cada cidade da pergunta aparece no SQL?──▶
        ──executa em conexão READ-ONLY──▶ linhas
        ──erro? devolve o erro ao LLM e tenta de novo (1x)──▶
        ──LLM responde só com o resultado──▶ resposta [banco: viagens.db]
```

In [4]:
from viagens.sql import TextToSQL
t2s = TextToSQL(llm)
print(t2s.schema[:1200], "...")


CREATE TABLE aeroportos (
	iata TEXT, 
	cidade TEXT NOT NULL, 
	pais TEXT NOT NULL, 
	PRIMARY KEY (iata)
)

/*
2 rows from aeroportos table:
iata	cidade	pais
GRU	São Paulo	Brasil
GIG	Rio de Janeiro	Brasil
*/


CREATE TABLE hoteis (
	id INTEGER, 
	nome TEXT NOT NULL, 
	cidade TEXT NOT NULL, 
	bairro TEXT, 
	estrelas INTEGER NOT NULL, 
	diaria_brl REAL NOT NULL, 
	nota REAL NOT NULL, 
	cafe_incluso INTEGER NOT NULL, 
	opcao_vegana INTEGER NOT NULL, 
	PRIMARY KEY (id)
)

/*
2 rows from hoteis table:
id	nome	cidade	bairro	estrelas	diaria_brl	nota	cafe_incluso	opcao_vegana
1	Hotel Alfama Azul	Lisboa	Alfama	3	546.54	8.0	1	0
2	Lisboa Tejo Palace	Lisboa	Baixa	5	1765.91	8.3	0	0
*/


CREATE TABLE reservas (
	id INTEGER, 
	usuario_id TEXT NOT NULL, 
	tipo TEXT NOT NULL, 
	voo_id INTEGER, 
	hotel_id INTEGER, 
	data_inicio TEXT NOT NULL, 
	data_fim TEXT, 
	pessoas INTEGER NOT NULL, 
	valor_total_brl REAL NOT NULL, 
	status TEXT NOT NULL, 
	criada_em TEXT NOT NULL, 
	PRIMARY KEY (id), 
	FOREIGN KEY

In [5]:
perguntas_sql = [
    "Qual o voo direto mais barato do Rio para Paris em abril de 2027?",
    "Quais hotéis em Paris têm opção vegana e nota acima de 8?",
    "Quais hotéis em Lisboa têm opção vegana e nota acima de 8?",
    "Qual a média de preço dos voos de São Paulo para Buenos Aires em julho e em março?",
    "Quais são as minhas reservas e o status de cada uma?",
    "Apague todas as reservas canceladas",
]
regs_sql = [t2s.consultar(q, config=CFG) for q in perguntas_sql]
for r in regs_sql:
    print("Pergunta:", r["pergunta"])
    print("SQL:     ", r["sql"])
    if any(t["erro"] for t in r["tentativas"]):
        print("Tentativas:", [t["erro"] for t in r["tentativas"]])
    print("Resposta:", r["resposta"][:350], "\n")

Pergunta: Qual o voo direto mais barato do Rio para Paris em abril de 2027?
SQL:      SELECT numero, data_partida, hora_partida, preco_brl FROM voos WHERE origem = 'GIG' AND destino = 'CDG' AND escalas = 0 AND classe = 'economica' AND data_partida BETWEEN '2027-04-01' AND '2027-04-30' ORDER BY preco_brl LIMIT 1
Resposta: O voo direto mais barato do Rio para Paris em abril de 2027 é o número AS2075, com partida no dia 30 de abril às 23:15, custando R$ 4.398,30. [banco: viagens.db] 

Pergunta: Quais hotéis em Paris têm opção vegana e nota acima de 8?
SQL:      SELECT * FROM hoteis WHERE cidade = 'Paris' AND nota > 8 AND opcao_vegana = 1 LIMIT 20
Tentativas: ["NAO_EXISTE recusado: o esquema tem ['hoteis.nota', 'hoteis.opcao_vegana']", None]
Resposta: Os hotéis em Paris que têm opção vegana e nota acima de 8 são: 

1. Hôtel Petit Marais - R$ 387,28, nota 8.9
2. Saint-Germain Rive - R$ 712,25, nota 8.9

[banco: viagens.db] 

Pergunta: Quais hotéis em Lisboa têm opção vegana e nota acima de 

### Salvaguardas do text-to-query, e as falhas reais que as motivaram

Na primeira versão (sem a tabela IATA no prompt e sem a validação de coerência), a execução real com `gpt-4o-mini` produziu dois erros **silenciosos**, que não geram exceção e devolvem uma resposta confiante e errada:

1. *"voo mais barato do Rio para **Paris**"* → `destino = 'LIS'`: o modelo copiou o código do exemplo few-shot (Lisboa). A resposta afirmava "voo para Paris" com os dados de um voo para Lisboa.
2. *"média de preços para **Buenos Aires**"* → `destino = 'BUE'` (o código do aeroporto do banco é `EZE`) e o ano de 2026, já passado. O resultado veio vazio e a resposta disse "não há registros".

Além disso, *"Apague as reservas canceladas"* virou um `SELECT`, e a resposta "não há registros" era enganosa.

Numa execução seguinte, o modelo também combinou `AND` e `OR` **sem parênteses** (`... AND março OR julho`), apesar da regra no prompt. Isso faz o filtro de rota valer só para março.

As correções: (a) a tabela cidade=IATA no prompt, com o aviso de que os exemplos usam outras cidades; (b) a **validação semântica** `validar_coerencia`, que confere se cada cidade da pergunta aparece no SQL e se não há `AND`/`OR` misturados sem parênteses e, se houver erro, manda-o para a autocorreção; (c) a instrução `RECUSAR` para pedidos de escrita. A célula abaixo mostra cada barreira funcionando sobre os SQLs problemáticos.

In [6]:
from viagens.sql import validar_sql, validar_coerencia, executar_sql, SQLInseguroError, SQLIncoerenteError

testes = [
    ("coerência", "Qual o voo mais barato do Rio para Paris?",
     "SELECT * FROM voos WHERE origem = 'GIG' AND destino = 'LIS' ORDER BY preco_brl LIMIT 1"),
    ("coerência", "Média de preço para Buenos Aires",
     "SELECT AVG(preco_brl) FROM voos WHERE destino = 'BUE'"),
    ("coerência", "Média de São Paulo para Buenos Aires em março ou julho",
     "SELECT AVG(preco_brl) FROM voos WHERE origem = 'GRU' AND destino = 'EZE' AND data_partida LIKE '2027-03%' OR data_partida LIKE '2027-07%'"),
    ("segurança", "-", "DROP TABLE reservas"),
    ("segurança", "-", "SELECT * FROM voos; DELETE FROM reservas"),
    ("segurança", "-", "SELECT sql FROM sqlite_master"),
    ("segurança", "-", "UPDATE reservas SET status = 'confirmada'"),
]
for tipo, pergunta, sql in testes:
    try:
        s = validar_sql(sql)
        if pergunta != "-":
            validar_coerencia(pergunta, s)
        print("PASSOU   ", sql)
    except (SQLInseguroError, SQLIncoerenteError) as e:
        print(f"BLOQUEADO ({tipo}): {sql[:60]:<62} → {e}")

# defesa em profundidade: mesmo sem a validação textual, a conexão é somente leitura
try:
    executar_sql("DELETE FROM reservas")
except Exception as e:
    print("\nConexão read-only recusou a escrita:", type(e).__name__, "-", e)

BLOQUEADO (coerência): SELECT * FROM voos WHERE origem = 'GIG' AND destino = 'LIS'    → O SQL não filtra pelas cidades citadas na pergunta: Paris (IATA CDG)
BLOQUEADO (coerência): SELECT AVG(preco_brl) FROM voos WHERE destino = 'BUE'          → O SQL não filtra pelas cidades citadas na pergunta: Buenos Aires (IATA EZE)
BLOQUEADO (coerência): SELECT AVG(preco_brl) FROM voos WHERE origem = 'GRU' AND des   → AND e OR no mesmo nível do WHERE sem parênteses: agrupe as alternativas do OR entre parênteses para não perder os demais filtros.
BLOQUEADO (segurança): DROP TABLE reservas                                            → Somente consultas SELECT são permitidas.
BLOQUEADO (segurança): SELECT * FROM voos; DELETE FROM reservas                       → Mais de uma instrução SQL não é permitida.
BLOQUEADO (segurança): SELECT sql FROM sqlite_master                                  → Acesso a tabelas de sistema não é permitido.
BLOQUEADO (segurança): UPDATE reservas SET status = 'confirmada'    

## 4. Agente de recuperação sobre o índice FAISS existente

O índice foi construído **uma única vez** na Fase 2 (`data/faiss_index/<modo>/`). Aqui ele é apenas carregado com `FAISS.load_local`, sem reprocessar documentos nem recalcular embeddings. O agente usa `create_retriever_tool`, com um `document_prompt` que prefixa cada trecho com `[fonte: arquivo.md]`, e `create_tool_calling_agent` + `AgentExecutor(return_intermediate_steps=True)`. Assim, os documentos-fonte que o agente **efetivamente recebeu** ficam disponíveis para diagnóstico.

In [7]:
from viagens.conhecimento import INDEX_DIR, carregar_indice
from viagens.recuperacao import (Buscador, CASOS_DIFICEIS, RetrieverTransformado, avaliar,
                                 criar_agente_recuperacao)

arquivos = {p.name: f"{p.stat().st_size / 1024:.0f} KB, gerado em {time.strftime('%Y-%m-%d %H:%M', time.localtime(p.stat().st_mtime))}"
            for p in INDEX_DIR.iterdir()}
print("Índice existente:", INDEX_DIR.relative_to(ROOT), arquivos)
vs = carregar_indice()
print("Vetores no índice:", vs.index.ntotal, "| sem reindexação nesta fase")

buscador = Buscador(llm, k=3)
retriever_base = vs.as_retriever(search_kwargs={"k": 3})
agente_rag = criar_agente_recuperacao(llm, retriever_base)

Índice existente: data/faiss_index/openai {'index.faiss': '186 KB, gerado em 2026-11-12 20:40', 'index.pkl': '19 KB, gerado em 2026-11-12 20:40'}


Vetores no índice: 31 | sem reindexação nesta fase


In [8]:
import re

def fontes_do_agente(res):
    """Fontes que a ferramenta devolveu ao agente (lidas da observação, não da resposta)."""
    return [f for _, obs in res["intermediate_steps"] for f in re.findall(r"\[fonte: ([^\]]+)\]", str(obs))]

for q in ["Qual o valor da ETA?", "Dá pra levar carregador portátil no avião?"]:
    r = agente_rag.invoke({"input": q}, config=CFG)
    print(f"P: {q}\n   fontes recebidas: {fontes_do_agente(r)}\n   R: {r['output'][:300]}\n")

P: Qual o valor da ETA?
   fontes recebidas: ['reino_unido_eta.md', 'schengen_regras.md', 'guia_barcelona.md']
   R: O valor da ETA (Electronic Travel Authorisation) para brasileiros é de 16 libras esterlinas por pessoa, incluindo crianças. Essa autorização deve ser solicitada antes do embarque e é válida por 2 anos ou até o vencimento do passaporte [fonte: reino_unido_eta.md].



P: Dá pra levar carregador portátil no avião?
   fontes recebidas: ['bagagem_aerosul.md', 'saude_seguro.md', 'bagagem_aerosul.md']
   R: Sim, é permitido levar carregadores portáteis (power banks) no avião, mas eles devem ser transportados na bagagem de mão e nunca na bagagem despachada [fonte: bagagem_aerosul.md].



## 5. Casos difíceis e diagnóstico pelos documentos-fonte

São 16 casos em dois grupos:
- **7 casos com cidade, e não país:** "documentos para Milão", "visto pra Roma", "Amsterdã"... A regra está em `schengen_regras.md`, escrito por país e bloco.
- **9 controles com vocabulário informal ou indireto:** "na gringa", "carregador portátil", "Bariloche", "Disney"... Eles precisam continuar funcionando depois da correção.

As métricas são calculadas sobre o top-3 devolvido pelo índice: **hit@1** (a fonte certa em 1º lugar), **hit@3** (a fonte certa entre as 3) e **MRR** (média de 1/posição da primeira fonte certa).

In [9]:
base = pd.DataFrame(avaliar(buscador.base, config=CFG))
print(f"BASELINE  hit@1={base['hit@1'].mean():.2f}  hit@3={base['hit@k'].mean():.2f}  MRR={base['rr'].mean():.2f}\n")
base.assign(fontes=base["fontes"].map(lambda f: [x.replace(".md", "") for x in f]))[["pergunta", "esperado", "fontes", "hit@k"]]

BASELINE  hit@1=0.50  hit@3=0.56  MRR=0.53



,pergunta,esperado,fontes,hit@k
0,Quais documentos preciso para entrar em Barcelona?,[schengen_regras.md],"[portugal_entrada, guia_barcelona, portugal_entrada]",False
1,O que preciso para entrar em Paris?,[schengen_regras.md],"[guia_paris, guia_paris, portugal_entrada]",False
2,Quais documentos levar para Milão?,[schengen_regras.md],"[portugal_entrada, menores_viagem, mercosul_documentos]",False
3,Brasileiro precisa de visto pra Roma?,[schengen_regras.md],"[eua_visto, guia_roma, portugal_entrada]",False
4,"Vou para Barcelona e depois Paris, quantos dias posso ficar?",[schengen_regras.md],"[guia_barcelona, guia_paris, guia_paris]",False
5,Que documentos preciso apresentar na imigração em Madri?,[schengen_regras.md],"[portugal_entrada, portugal_entrada, mercosul_documentos]",False
6,"Vou para Amsterdã, preciso de visto?",[schengen_regras.md],"[eua_visto, eua_visto, portugal_entrada]",False
7,Preciso fazer algum cadastro online antes de ir pra Inglaterra?,[reino_unido_eta.md],"[reino_unido_eta, reino_unido_eta, schengen_regras]",True
8,"Meu filho de 15 anos vai pra Disney com a avó, o que precisa?","[menores_viagem.md, eua_visto.md]","[menores_viagem, menores_viagem, saude_seguro]",True
9,Quanto de imposto pago usando o cartão na gringa?,[dinheiro_iof.md],"[dinheiro_iof, dinheiro_iof, guia_barcelona]",True


In [10]:
# Diagnóstico: o que o índice devolveu nas falhas (fonte, distância e começo do trecho)
falhas = base[~base["hit@k"]]["pergunta"].tolist()
for q in falhas:
    print(f"✘ {q}")
    for d, s in vs.similarity_search_with_score(q, k=3):
        print(f"     {float(s):.3f}  {d.metadata['fonte']:<22} {d.page_content[:85]!r}")
    print()

✘ Quais documentos preciso para entrar em Barcelona?


     0.832  portugal_entrada.md    '## Documentos exigidos na fronteira\n- Passaporte válido por pelo menos 3 meses após a'
     0.835  guia_barcelona.md      '# Barcelona, Espanha\n\nMoeda: euro (EUR). Idiomas: catalão e espanhol. Tomadas: tipo F'
     1.003  portugal_entrada.md    '## Observações\nO RG brasileiro não é aceito para entrar em Portugal: é obrigatório o '

✘ O que preciso para entrar em Paris?


     0.904  guia_paris.md          '# Paris, França\n\nMoeda: euro (EUR). Idioma: francês. Tomadas: tipo E, 230 V.\n\n## Clim'
     0.972  guia_paris.md          '## Destaques e reservas\nTorre Eiffel, Louvre (reserva de horário obrigatória), Museu '
     0.972  portugal_entrada.md    '## Documentos exigidos na fronteira\n- Passaporte válido por pelo menos 3 meses após a'

✘ Quais documentos levar para Milão?


     0.963  portugal_entrada.md    '## Documentos exigidos na fronteira\n- Passaporte válido por pelo menos 3 meses após a'
     1.104  menores_viagem.md      '## Formas de autorização\n- Formulário de autorização de viagem com firma reconhecida '
     1.110  mercosul_documentos.md '# América do Sul: viajar só com documento de identidade\n\nGraças aos acordos do Mercos'

✘ Brasileiro precisa de visto pra Roma?


     0.912  eua_visto.md           '# Estados Unidos: visto de turista (B1/B2)\n\nBrasileiros precisam de visto para entrar'
     0.920  guia_roma.md           '# Roma, Itália\n\nMoeda: euro (EUR). Idioma: italiano. Tomadas: tipos F e L, 230 V.\n\n##'
     0.950  portugal_entrada.md    '## Observações\nO RG brasileiro não é aceito para entrar em Portugal: é obrigatório o '

✘ Vou para Barcelona e depois Paris, quantos dias posso ficar?


     0.888  guia_barcelona.md      '# Barcelona, Espanha\n\nMoeda: euro (EUR). Idiomas: catalão e espanhol. Tomadas: tipo F'
     0.967  guia_paris.md          '# Paris, França\n\nMoeda: euro (EUR). Idioma: francês. Tomadas: tipo E, 230 V.\n\n## Clim'
     1.032  guia_paris.md          '## Destaques e reservas\nTorre Eiffel, Louvre (reserva de horário obrigatória), Museu '

✘ Que documentos preciso apresentar na imigração em Madri?
     0.826  portugal_entrada.md    '## Documentos exigidos na fronteira\n- Passaporte válido por pelo menos 3 meses após a'
     1.047  portugal_entrada.md    '## Observações\nO RG brasileiro não é aceito para entrar em Portugal: é obrigatório o '
     1.060  mercosul_documentos.md '# América do Sul: viajar só com documento de identidade\n\nGraças aos acordos do Mercos'

✘ Vou para Amsterdã, preciso de visto?


     1.071  eua_visto.md           '## Etapas\n1. Preencher o formulário DS-160 no site do Departamento de Estado.\n2. Paga'
     1.103  eua_visto.md           '# Estados Unidos: visto de turista (B1/B2)\n\nBrasileiros precisam de visto para entrar'
     1.105  portugal_entrada.md    '## Documentos exigidos na fronteira\n- Passaporte válido por pelo menos 3 meses após a'



In [11]:
# Onde estão as palavras-chave dessas perguntas na base?
from viagens.conhecimento import carregar_documentos
docs = carregar_documentos()
for termo in ["Barcelona", "Paris", "Milão", "Roma", "Madri", "Amsterdã", "Espanha", "Itália", "França", "Holanda", "Schengen"]:
    onde = [d.metadata["fonte"] for d in docs if termo.lower() in d.page_content.lower()]
    print(f"{termo:<10} aparece em: {onde or '— nenhum documento'}")

Barcelona  aparece em: ['guia_barcelona.md']
Paris      aparece em: ['guia_paris.md']
Milão      aparece em: — nenhum documento
Roma       aparece em: ['guia_roma.md']
Madri      aparece em: — nenhum documento
Amsterdã   aparece em: — nenhum documento
Espanha    aparece em: ['guia_barcelona.md', 'schengen_regras.md']
Itália     aparece em: ['guia_roma.md', 'schengen_regras.md']
França     aparece em: ['guia_paris.md', 'schengen_regras.md']
Holanda    aparece em: ['schengen_regras.md']
Schengen   aparece em: ['portugal_entrada.md', 'reino_unido_eta.md', 'saude_seguro.md', 'schengen_regras.md']


### Diagnóstico (execução real, embeddings `text-embedding-3-small`)

As 7 falhas do baseline são **todas** perguntas que citam uma **cidade** europeia, e o padrão das fontes devolvidas explica a causa:

1. **Descompasso de vocabulário entre a pergunta e o texto indexado.** As regras de entrada estão escritas por **país e bloco** ("Espanha", "Itália", "Espaço Schengen"). As cidades aparecem só nos guias turísticos, e Milão e Amsterdã não aparecem em documento nenhum (célula acima). O embedding da pergunta é dominado pelo que ela tem de genérico ("documentos", "entrar", "visto") mais o nome da cidade.
2. **Documentos "parecidos na forma, errados no conteúdo" vencem.** `portugal_entrada.md` tem o título "Entrada em Portugal" e a seção "Documentos exigidos na fronteira", muito parecidos com "documentos para entrar em X", mas fala de **outro país**. Pior: em "visto pra Roma" e "Amsterdã, preciso de visto?", o primeiro lugar foi `eua_visto.md`, o visto **americano**, porque a palavra "visto" pesa mais que a cidade, que o embedding não liga a "Schengen".
3. **O guia da cidade tem o nome certo, mas não tem a regra.** `guia_barcelona.md` e `guia_paris.md` aparecem por causa do nome da cidade, mas não falam de documentos de entrada. Foi isso que, na Fase 2, levou a síntese a citar `guia_barcelona.md` como fonte de uma regra que ele não contém.

A ligação que falta (Milão → Itália → Espaço Schengen) é **conhecimento de mundo**, que o LLM tem e a busca vetorial não aplica. A correção, portanto, deve acontecer **antes** da busca, na consulta, e não no índice.

## 6. Transformação de consulta: três técnicas comparadas

| Técnica | Como funciona | Custo extra por pergunta |
|---|---|---|
| **Reescrita** | O LLM reescreve a pergunta no vocabulário da base: cidade → país + bloco, gíria → termo oficial | 1 chamada curta + 1 busca |
| **Multi-query** | O LLM gera 3 variações; as 4 consultas (com a original) são unidas por *Reciprocal Rank Fusion* | 1 chamada + 4 buscas |
| **HyDE** | O LLM escreve um trecho hipotético de guia que responderia à pergunta; o trecho é a consulta | 1 chamada longa + 1 busca |

In [12]:
resultados, custos = {}, {}
for nome in ["base", "reescrita", "multi_query", "hyde"]:
    h = LogExecucaoHandler(sessao=f"rag_{nome}")      # handler próprio: mede os tokens de cada técnica
    t0 = time.perf_counter()
    resultados[nome] = pd.DataFrame(avaliar(buscador.estrategia(nome), config={"callbacks": [h, log_handler]}))
    n = len(resultados[nome])
    ev_h = h.resumo()
    tokens = int(ev_h["tokens"].fillna(0).sum()) if len(ev_h) and "tokens" in ev_h else 0
    custos[nome] = {"latência média (s)": (time.perf_counter() - t0) / n, "tokens LLM por pergunta": tokens / n}

grupo = lambda df, a, b: df.iloc[a:b]
comparacao = pd.DataFrame({
    nome: {"hit@1": df["hit@1"].mean(), "hit@3": df["hit@k"].mean(), "MRR": df["rr"].mean(),
           "hit@3 (7 casos de cidade)": grupo(df, 0, 7)["hit@k"].mean(),
           "hit@3 (9 controles)": grupo(df, 7, 16)["hit@k"].mean(), **custos[nome]}
    for nome, df in resultados.items()}).T.round(2)
comparacao

,hit@1,hit@3,MRR,hit@3 (7 casos de cidade),hit@3 (9 controles),latência média (s),tokens LLM por pergunta
base,0.50,0.56,0.53,0.00,1.0,0.29,0.00
reescrita,0.75,1.00,0.88,1.00,1.0,1.37,216.31
multi_query,0.62,0.75,0.68,0.43,1.0,2.28,131.69
hyde,0.75,1.00,0.86,1.00,1.0,3.10,198.75


In [13]:
# O que a reescrita fez com as perguntas que falhavam (consulta transformada → fontes)
for q in falhas:
    docs_r = buscador.reescrita(q, config=CFG)
    print(f"• {q}\n   → {buscador.ultima_consulta['consultas'][0]}\n   fontes: {[d.metadata['fonte'] for d in docs_r]}\n")

• Quais documentos preciso para entrar em Barcelona?
   → Quais documentos um brasileiro precisa para entrar na Espanha, Espaço Schengen?
   fontes: ['portugal_entrada.md', 'schengen_regras.md', 'portugal_entrada.md']



• O que preciso para entrar em Paris?
   → O que preciso para entrar na França, Espaço Schengen?
   fontes: ['portugal_entrada.md', 'schengen_regras.md', 'schengen_regras.md']



• Quais documentos levar para Milão?
   → Quais documentos levar para a Itália, Espaço Schengen?
   fontes: ['portugal_entrada.md', 'schengen_regras.md', 'schengen_regras.md']



• Brasileiro precisa de visto pra Roma?
   → Brasileiro precisa de visto para a Itália, Espaço Schengen?
   fontes: ['schengen_regras.md', 'portugal_entrada.md', 'portugal_entrada.md']



• Vou para Barcelona e depois Paris, quantos dias posso ficar?
   → Sou brasileiro e vou para a Espanha e depois para a França, quantos dias posso ficar?
   fontes: ['schengen_regras.md', 'portugal_entrada.md', 'mercosul_documentos.md']



• Que documentos preciso apresentar na imigração em Madri?
   → Quais documentos um brasileiro precisa apresentar na imigração na Espanha, Espaço Schengen?
   fontes: ['portugal_entrada.md', 'schengen_regras.md', 'portugal_entrada.md']



• Vou para Amsterdã, preciso de visto?
   → Sou brasileiro e vou para a Holanda, preciso de visto para o Espaço Schengen?
   fontes: ['schengen_regras.md', 'portugal_entrada.md', 'portugal_entrada.md']



### Escolha: **reescrita de consulta**

A tabela acima traz o resultado desta execução. Os fatos que se repetiram nas execuções reais:

- **Reescrita e HyDE corrigem os 7 casos de cidade** (o hit@3 nesse grupo sai de 0,00), e **nenhuma das duas quebra os controles**. As duas atacam a causa diagnosticada: aplicam o conhecimento de mundo (Milão → Itália → Espaço Schengen) que a busca vetorial não tem.
- **Multi-query corrige pouco.** A consulta original entra na fusão e continua puxando `portugal_entrada.md` e `eua_visto.md`. Variar a formulação ajuda em perguntas ambíguas, não quando o vocabulário é sistematicamente diferente do da base.
- **HyDE e reescrita empatam no hit@3**, e o HyDE às vezes tem hit@1 e MRR um pouco maiores. Mas, numa execução exploratória anterior, o HyDE errou "documentos para Milão" (hit@3 = 0,94), enquanto a reescrita acertou os 16 casos nas duas execuções.

**Por que a reescrita, e não o HyDE:**
1. **Latência** (colunas da tabela): o consumo de tokens das duas é parecido, porque o prompt da reescrita é mais longo e o HyDE gera mais texto. Mas a reescrita gera só uma frase e responde em cerca de metade do tempo, o que conta porque a recuperação roda em toda pergunta de regras.
2. **Estabilidade:** acertou 16/16 nas duas execuções reais.
3. **Auditabilidade:** a consulta reescrita cabe numa linha e pode ser verificada ("Itália, Espaço Schengen"). O trecho do HyDE é um texto **inventado** pelo modelo, que pode trazer "fatos" falsos para a consulta. Num sistema que depois verifica a resposta contra as fontes (Fase 5), é melhor que a consulta não fabrique conteúdo.

**O que ainda não está resolvido.** Mesmo com a reescrita, `portugal_entrada.md` continua aparecendo no top-3 de perguntas sobre Espanha e Itália, porque ele também fala de "Espaço Schengen". O agente recebe a regra certa, mas também um documento de outro país (seção 7). A evolução natural é um **filtro por metadado** (`paises` de cada documento), com o país extraído na própria reescrita (*self-query*).

No modo dummy (embedding lexical), a reescrita é imitada por um glossário fixo, e as tabelas mostram o mesmo efeito.

## 7. O agente de recuperação corrigido

O `RetrieverTransformado` é um `BaseRetriever` que reescreve a consulta antes de buscar no **mesmo índice**. O agente é o mesmo da seção 4; só o retriever da ferramenta muda.

In [14]:
agente_corrigido = criar_agente_recuperacao(llm, RetrieverTransformado(buscador=buscador, nome_estrategia="reescrita"))

for q in ["Quais documentos preciso para entrar em Barcelona?", "Brasileiro precisa de visto pra Roma?",
          "Vou para Amsterdã, preciso de visto?"]:
    antes = agente_rag.invoke({"input": q}, config=CFG)
    depois = agente_corrigido.invoke({"input": q}, config=CFG)
    print(f"P: {q}")
    print(f"   ANTES  fontes: {sorted(set(fontes_do_agente(antes)))}\n          R: {antes['output'][:260]}")
    print(f"   DEPOIS fontes: {sorted(set(fontes_do_agente(depois)))}\n          R: {depois['output'][:260]}\n")

P: Quais documentos preciso para entrar em Barcelona?
   ANTES  fontes: ['guia_barcelona.md', 'portugal_entrada.md']
          R: Para entrar em Barcelona, você precisará dos seguintes documentos: passaporte válido por pelo menos 3 meses após a data prevista de saída do Espaço Schengen, passagem de volta ou de continuação da viagem, comprovante de hospedagem, seguro-viagem com cobertura 
   DEPOIS fontes: ['portugal_entrada.md', 'schengen_regras.md']
          R: Para entrar em Barcelona, você precisará de um passaporte válido por pelo menos 3 meses após a data prevista de saída do Espaço Schengen, passagem de volta, comprovante de hospedagem, seguro-viagem com cobertura mínima de 30.000 euros e comprovante de meios de



P: Brasileiro precisa de visto pra Roma?
   ANTES  fontes: ['guia_lisboa.md', 'guia_roma.md']
          R: A base não cobre o caso sobre a necessidade de visto para brasileiros que desejam viajar para Roma.
   DEPOIS fontes: ['portugal_entrada.md', 'schengen_regras.md']
          R: Brasileiros não precisam de visto para estadias de turismo ou negócios de até 90 dias em Roma, que faz parte do Espaço Schengen. É importante observar que esse limite de 90 dias se aplica ao conjunto dos países Schengen em um período de 180 dias [fonte: scheng



P: Vou para Amsterdã, preciso de visto?
   ANTES  fontes: ['eua_visto.md', 'guia_lisboa.md', 'guia_roma.md']
          R: A base não cobre o caso sobre a necessidade de visto para Amsterdã.
   DEPOIS fontes: ['portugal_entrada.md', 'schengen_regras.md']
          R: Brasileiros não precisam de visto para estadias de turismo ou negócios de até 90 dias em Amsterdã, que faz parte do Espaço Schengen. É importante lembrar que esse limite de 90 dias se aplica ao total de dias em todos os países Schengen dentro de um período de 



## 8. O agente com as novas ferramentas

Um agente de *tool calling* com as ferramentas desta fase (text-to-SQL, API interna, serverless) mais as anteriores. Ele precisa escolher o **tipo** certo de integração para cada parte do pedido.

In [15]:
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate

prompt_ag = ChatPromptTemplate.from_messages([
    ("system", "Você é um assistente de viagens. Use consultar_banco para voos, hotéis e reservas no banco; "
               "listar_reservas/criar_reserva para a API interna de reservas; estimar_custo para o custo total; "
               "converter_moeda e previsao_tempo para dados externos. Responda em até 5 frases, citando as fontes."),
    ("human", "{input}"), ("placeholder", "{agent_scratchpad}")])
agente_f4 = AgentExecutor(agent=create_tool_calling_agent(llm, tools, prompt_ag), tools=tools,
                          return_intermediate_steps=True, max_iterations=6)

for q in ["Qual o voo direto mais barato de São Paulo para Lisboa em março de 2027? Quero o número e o preço.",
          "Quanto custaria no total uma viagem de 10 dias para 2 pessoas com passagem de 4.500 reais por pessoa "
          "e hotel de 700 reais a diária?",
          "Quais são as minhas reservas na API de reservas?"]:
    r = agente_f4.invoke({"input": q}, config=CFG)
    print("P:", q)
    for a, _ in r["intermediate_steps"]:
        print("   →", a.tool, json.dumps(a.tool_input, ensure_ascii=False)[:150])
    print("   R:", r["output"][:400], "\n")

P: Qual o voo direto mais barato de São Paulo para Lisboa em março de 2027? Quero o número e o preço.
   → consultar_banco {"pergunta": "voo direto mais barato de São Paulo para Lisboa em março de 2027"}
   R: O voo direto mais barato de São Paulo para Lisboa em março de 2027 é o número AS1105, com partida no dia 22 de março às 23:15, custando R$ 3.417,59. 



P: Quanto custaria no total uma viagem de 10 dias para 2 pessoas com passagem de 4.500 reais por pessoa e hotel de 700 reais a diária?
   → estimar_custo {"dias": 10, "pessoas": 2, "passagem_por_pessoa_brl": 4500, "diaria_hotel_brl": 700}
   R: O custo total da viagem de 10 dias para 2 pessoas seria de R$ 29.430,00. Isso inclui R$ 9.000,00 em passagens, R$ 7.000,00 em hospedagem e R$ 11.000,00 em gastos locais, além de R$ 630,00 de IOF e R$ 1.800,00 de margem cambial. Cada pessoa pagaria aproximadamente R$ 14.715,00. Os cálculos consideram um gasto diário de R$ 550 por pessoa (perfil moderado). 



P: Quais são as minhas reservas na API de reservas?
   → listar_reservas {}
   R: Você possui as seguintes reservas:

1. **Voo**: AS1015 de GRU para LIS, confirmada para 16 de dezembro de 2026, para 2 pessoas, totalizando R$ 9.172,14.
2. **Hotel**: Lisboa Tejo Palace, confirmada de 16 a 23 de dezembro de 2026, para 2 pessoas, totalizando R$ 12.361,37.
3. **Voo**: AS1640 de OPO para GIG, pendente para 17 de fevereiro de 2027, para 1 pessoa, totalizando R$ 2.916,74.
4. **Voo**: A 



## 9. Decisões técnicas e limitações

| Decisão | Justificativa |
|---|---|
| Text-to-SQL próprio (prompt + validações), usando `SQLDatabase` para o schema | O `create_sql_query_chain` gera o SQL, mas não valida coerência nem segurança. Os dois erros silenciosos observados (IATA copiado e IATA inventado) exigiram uma validação **semântica**, além da sintática. |
| Conexão SQLite read-only **e** validação textual | Defesa em profundidade: a validação dá uma mensagem útil, e a conexão garante a segurança mesmo que a validação seja burlada. |
| Escrita só pela API interna | A regra de negócio (assentos, limites, status `pendente`) vive no serviço dono dos dados, não no prompt. |
| Índice FAISS só carregado | Atende ao requisito e evita custo de embeddings. A correção foi na consulta, não no índice. |
| Reescrita como técnica de transformação | Corrige os 7 casos sem quebrar controles, empata com o HyDE no hit@3 com menor custo e foi estável nas execuções, além de produzir uma consulta auditável (seção 6). |

**Limitações:**
- O banco e a companhia aérea são fictícios. A validação de coerência só conhece as cidades da tabela `aeroportos`.
- A reescrita adiciona cerca de 1 s de latência por busca. Um cache de consultas frequentes reduziria esse custo.
- 16 casos de teste mostram a tendência, mas não têm significância estatística. A Fase 5 amplia a avaliação no LangSmith.
- Mesmo com a reescrita, documentos de outro país do Schengen (`portugal_entrada.md`) ainda entram no top-3. Um filtro por metadado de país resolveria.
- Na execução real, o text-to-SQL respondeu "a média de julho e março é R$ 1.400" quando só havia voos em março (um único `AVG` com `OR`). As regras de `GROUP BY` em comparações e de declarar períodos sem dados foram adicionadas depois disso.

In [16]:
ev = log_handler.resumo()
print("Eventos:", len(ev), "|", ev["evento"].value_counts().to_dict())
if "tokens" in ev and ev["tokens"].notna().any():
    print("Tokens consumidos:", int(ev["tokens"].fillna(0).sum()))
reservas_api.parar_servico()

Eventos: 149 | {'llm_fim': 95, 'tool_inicio': 16, 'tool_fim': 16, 'agente_acao': 11, 'agente_fim': 11}
Tokens consumidos: 24327
